In [50]:
class Config:

    #Globals
     #Globals
    batch_size = 16
    num_classes = 3  # classes, seizure/no seizure
    epochs = 25   # Epoch iterations
    time_step_length = 5
    row_hidden = 128  # hidden neurons in conv layers
    col_hidden = 128   # hidden neurons in the Bi-LSTM layers
    RANDOM_SEED = 3333    
    N_TIME_STEPS = 125   # 50 records in each sequence
    N_FEATURES = 3     # mag,hr,roi_Ratio,output
    step = 100           # window overlap = 50 -10 = 40  (80% overlap)
    N_CLASSES = 3      # class label
    learning_rate = 0.0000001
    k = 5 # number of k folds
    target_class_count=58250

In [1]:
import numpy as np 
import pandas as pd 
import json 

class IpdDataLoader: 
    def __init__(self, file_path, time_steps):
        self.file_path = file_path
        self.time_steps = time_steps
        self.df_sensordata = None
        self.load_and_process_data_from_json()

    def load_and_process_data_from_json(self):
        """
        Load and process OSDB data from a JSON file. This function will create a DataFrame 
        with the necessary columns and calculate FFT features.
        """
        with open(self.file_path, 'r') as file:
            raw_json = json.load(file)

        # Flatten the JSON and extract the necessary data
        flattened_data = []
        for attribute in raw_json:
            user_id = attribute.get('userId', None)
            datapoints = attribute.get('datapoints', [])

            for point in datapoints:
                event_id = point.get('eventId', None)
                hr = point.get('hr', None)
                o2Sat = point.get('o2Sat', None)
                rawData = point.get('rawData', [])
                rawData3D = point.get('rawData3D', [])

                # FFT calculation for rawData
                fft_result = self.calculate_fft(rawData)
                #Uncomment the sensor data that you want to load
                flattened_data.append({
                    'eventId': event_id,
                    'userId': user_id,
                    'hr': hr,
                    #'o2Sat': o2Sat,
                    'rawData': rawData,
                    #'rawData3D': rawData3D,
                    'FFT': fft_result  # Adding FFT column directly
                })

        # Create DataFrame from the flattened data
        self.df_sensordata = pd.DataFrame(flattened_data)

        # Apply zero padding to the FFT column to make sure all rows have 125 FFT values
        self.df_sensordata['FFT'] = self.df_sensordata['FFT'].apply(lambda fft: np.pad(fft, (0, 125 - len(fft)), 'constant', constant_values=0) if len(fft) < 125 else fft)

    def calculate_fft(self, raw_data):
        if not raw_data:
            return []

        # Convert raw_data to numpy array
        raw_data = np.array(raw_data)
        # Perform FFT, remove DC component, and return magnitudes
        raw_data = raw_data - np.mean(raw_data)  # Remove DC component
        fft_result = np.fft.fft(raw_data)
        fft_magnitude = np.abs(fft_result)
        # Isolate positive frequencies
        positive_fft_magnitude = fft_magnitude[:len(fft_magnitude) // 2]
        
        return positive_fft_magnitude.tolist()  # Return as a list
    
    

In [2]:
class IpdDataReshaper:
    def __init__(self, dataframe):
        self.df = dataframe

    def reshape_data(self):
        reshaped_rows = []
        
        for idx, row in self.df.iterrows():
            event_id = row['eventId']
            user_id = row['userId']
            hr = row['hr']
            #o2Sat = row['o2Sat']
            rawData = row['rawData']
            #rawData3D = row['rawData3D']
            fft = row['FFT']
            
            # Replicate eventId, userId, hr, o2Sat for 125 times
            repeated_info = {
                'eventId': [event_id] * 125,
                'userId': [user_id] * 125,
                'hr': [hr] * 125,
                #'o2Sat': [o2Sat] * 125
            }
            
            # Transpose rawData and FFT
            rawData_transposed = rawData[:125]  # Transpose to the correct shape
            fft_transposed = fft[:125]  # Transpose to the correct shape
            
            # Process rawData3D if it exists
            #if rawData3D:
            #    # Convert rawData3D into lists of 3 (x, y, z)
            #     rawData3D_transposed = [rawData3D[i:i+3] for i in range(0, len(rawData3D), 3)]
            #    rawData3D_transposed = rawData3D_transposed[:125]  # Ensure only 125 rows
            #else:
            #    rawData3D_transposed = [None] * 125  # If no rawData3D, set it to None
            
            # Create the reshaped row
            for i in range(125):
                reshaped_rows.append({
                    'eventId': repeated_info['eventId'][i],
                    'userId': repeated_info['userId'][i],
                    'hr': repeated_info['hr'][i],
                    #'o2Sat': repeated_info['o2Sat'][i],
                    'rawData': rawData_transposed[i],
                    #'rawData3D': rawData3D_transposed[i],
                    'FFT': fft_transposed[i]
                })
        
        # Create a new DataFrame from the reshaped rows
        reshaped_df = pd.DataFrame(reshaped_rows)
        return reshaped_df

In [53]:
from scipy.interpolate import CubicSpline

class IpdInterpolator:
    def __init__(self, df, column_to_interpolate):
        """
        Initialize the Interpolator class with a DataFrame and the column to interpolate.
        """
        self.df = df
        self.column_to_interpolate = column_to_interpolate

    def interpolate_column(self, new_column_name='interpolated_hr', interval=125, time_step=5):
        """
        Interpolate the specified column using the provided logic.
        
        Parameters:
        - new_column_name: Name of the new column to store interpolated values.
        - interval: Interval to sample the original column (e.g., every 125th element).
        - time_step: Time step in seconds for the interpolation process.
        """
        try:
            # Step 1: Extract every nth element from the specified column
            original_values = self.df[self.column_to_interpolate]
            selected_elements = original_values[0::interval]
            x = np.array(selected_elements)

            # Step 2: Create an array representing the time (in `time_step` intervals)
            time_values = np.arange(len(x)) * time_step

            # Step 3: Create a CubicSpline object for interpolation
            cs = CubicSpline(time_values, x, bc_type='clamped')

            # Step 4: Generate new time values for finer granularity
            num_original_points = len(x)
            new_time_values = np.linspace(0, (num_original_points - 1) * time_step, num_original_points * interval)

            # Step 5: Generate interpolated values
            interpolated_values = cs(new_time_values)

            # Step 6: Update the DataFrame with the interpolated values
            self.df[new_column_name] = interpolated_values[:len(self.df)]  # Match the original DataFrame length

            print(f"Interpolation completed. New column '{new_column_name}' added to the DataFrame.")
        except Exception as e:
            print("An error occurred during interpolation:", e)

    def get_dataframe(self):
        """
        Return the updated DataFrame with interpolated values.
        """
        return self.df

In [54]:
import pandas as pd
import numpy as np

class IpdDataProcessor:
    def __init__(self, data_file):
        """
        Initialize the class with the data file.
        Extract unique event IDs from the file and prepare data structures.
        """
        self.data_file = data_file
        self.df_labels = None
        self.df_sensor_data_filtered = None
        self.df_sensor_data = None
        self.ids = None  # To store unique event IDs
        
        # Load the data from the provided file
        self.load_data()
        self.extract_unique_ids()

    def load_data(self):
        """Load data from CSV and prepare the label DataFrame."""
        # Read the CSV file
        df = pd.read_csv(self.data_file)
        
        # Select only the required columns
        self.df_labels = df[["Id", "eventId", "label"]]
        
        # Sort the labels based on 'Id'
        self.df_labels = self.df_labels.sort_values(by='Id').reset_index(drop=True)

    def extract_unique_ids(self):
        """Extract unique event IDs from the label DataFrame."""
        # Extract unique event IDs and store them
        self.ids = self.df_labels['eventId'].unique()

    def filter_and_sort_sensor_data(self, interpolated_df):
        """Filter sensor data based on eventIds and sort the DataFrame."""
        # Filter the sensor data based on eventIds
        self.df_sensor_data_filtered = interpolated_df[interpolated_df['eventId'].isin(self.ids)]
        
        # Define 'eventId' as a categorical column with the desired order
        self.df_sensor_data_filtered['eventId'] = pd.Categorical(self.df_sensor_data_filtered['eventId'], categories=self.ids, ordered=True)
        
        # Sort the sensor data by 'eventId'
        self.df_sensor_data_filtered = self.df_sensor_data_filtered.sort_values('eventId')

    def merge_labels(self):
        """Merge the label column from df_labels to df_sensor_data_filtered based on eventId."""
        # Drop duplicates to ensure unique eventIds
        df_labels_unique = self.df_labels.drop_duplicates(subset='eventId', keep='first')
        
        # Merge the labels with the filtered sensor data
        self.df_sensor_data = pd.merge(self.df_sensor_data_filtered, df_labels_unique[['eventId', 'label']], on='eventId', how='left')

    def get_sensor_data(self):
        """Return the processed sensor data."""
        return self.df_sensor_data


In [55]:
import pandas as pd
import numpy as np
from collections import Counter

class TimeSeriesUndersampler:
    def __init__(self, timestep_size=125, prefix=999):
        """
        Initialize the TimeSeriesUndersampler.

        :param timestep_size: Number of rows in each timestep.
        :param prefix: Prefix to add to new eventIds for undersampled data.
        """
        self.timestep_size = timestep_size
        self.prefix = prefix

    def fit_resample(self, df, target_column, target_counts):
        """
        Perform undersampling on the given dataframe.

        :param df: Input dataframe.
        :param target_column: Column containing class labels.
        :param target_counts: Dictionary specifying target counts for each class.
        :return: Resampled dataframe.
        """
        resampled_df = []
        event_id_counter = Counter()

        for label, target_count in target_counts.items():
            label_df = df[df[target_column] == label]
            grouped = label_df.groupby('eventId')

            timesteps = []
            for event_id, group in grouped:
                timesteps.extend([group.iloc[i:i + self.timestep_size] for i in range(0, len(group), self.timestep_size)])

            np.random.shuffle(timesteps)  # Shuffle timesteps
            selected_timesteps = timesteps[:target_count // self.timestep_size]

            for timestep in selected_timesteps:
                if len(timestep) == self.timestep_size:
                    new_event_id = f"{self.prefix}{timestep['eventId'].iloc[0]}"
                    timestep['eventId'] = new_event_id
                    resampled_df.append(timestep)

            # Handle leftover rows for partial timesteps (if needed)
            leftover_count = target_count % self.timestep_size
            if leftover_count > 0:
                leftover_timesteps = timesteps[target_count // self.timestep_size:]
                leftover_rows = []
                for ts in leftover_timesteps:
                    if len(leftover_rows) >= leftover_count:
                        break
                    leftover_rows.extend(ts.to_dict('records'))
                leftover_df = pd.DataFrame(leftover_rows[:leftover_count])
                new_event_id = f"{self.prefix}{leftover_df['eventId'].iloc[0]}"
                leftover_df['eventId'] = new_event_id
                resampled_df.append(leftover_df)

        return pd.concat(resampled_df, ignore_index=True)

# Example usage
# Assuming df is your dataframe with 'label' as the target column
# And you want the following counts per class: {0: 92500, 1: 92500, 2: 92500}



In [64]:
from scipy import stats

class DataLoader:
    def __init__(self, dataframe, time_steps, step, target_column):
        self.dataframe = dataframe
        self.time_steps = time_steps
        self.step = step
        self.target_column = target_column

    def load_data(self):
        segments = []
        labels = []
        event_ids = []
        user_ids = []

        # Group data by eventID to ensure events are kept intact
        grouped = self.dataframe.groupby('eventId')

        for event_id, group in grouped:
            if len(group) >= self.time_steps:  # Process if the event group has enough data
                for i in range(0, len(group) - self.time_steps + 1, self.step):
                    mag = group['rawData'].values[i: i + self.time_steps]
                    hr = group['interpolated_hr'].values[i: i + self.time_steps]
                    fft = group['FFT'].values[i: i + self.time_steps]
                    segment = np.column_stack((hr, mag, fft))  # Combine magnitude and heart rate features
                    label_mode = stats.mode(group[self.target_column][i: i + self.time_steps])
                    if isinstance(label_mode.mode, np.ndarray):
                        label = label_mode.mode[0]
                    else:
                        label = label_mode.mode

                    segments.append(segment)
                    labels.append(label)
                    event_ids.append(event_id)
                    user_ids.append(group['userId'].iloc[0])  # Assuming userID is consistent within an event

        # Convert to numpy arrays
        segments = np.asarray(segments, dtype=np.float32)
        labels = np.asarray(pd.get_dummies(labels), dtype=np.float32)

        # Create DataFrame to store eventID and userID alongside segments and labels
        df_labels = pd.DataFrame({
            'segments': list(segments),
            'label': list(labels),
            'eventId': event_ids,
            'userId': user_ids
        })

        return df_labels

In [65]:
import pandas as pd
from sklearn.model_selection import train_test_split

class EventBasedSplitter:
    def __init__(self, dataframe, test_size=0.25, random_state=42):
        """
        Initialize the EventBasedSplitter class.
        
        :param dataframe: The input DataFrame containing the data to split.
        :param test_size: Proportion of eventIDs to include in the test set (default: 0.25).
        :param random_state: Random seed for reproducibility (default: 42).
        """
        self.df = dataframe
        self.test_size = test_size
        self.random_state = random_state

    def split_by_event(self):
        """
        Split the DataFrame into training and testing sets by eventID.
        
        :return: Two DataFrames - train_df and test_df.
        """
        # Extract unique eventIDs
        unique_event_ids = self.df['eventId'].unique()
        
        # Randomly split the eventIDs into train and test sets
        train_event_ids, test_event_ids = train_test_split(
            unique_event_ids, 
            test_size=self.test_size, 
            random_state=self.random_state
        )
        
        # Create training and testing datasets based on eventIDs
        train_df = self.df[self.df['eventId'].isin(train_event_ids)]
        test_df = self.df[self.df['eventId'].isin(test_event_ids)]
        
        return train_df, test_df


In [66]:
from tensorflow.keras.layers import (
    Layer, Add, Input, Conv1D, BatchNormalization, Activation,
    MaxPooling1D, Bidirectional, LSTM, Dense, Dropout,
    Reshape, Permute, Attention, GlobalMaxPooling1D, Concatenate, MultiHeadAttention
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.regularizers import l2
import tensorflow as tf
import json
from tensorflow.keras.models import save_model as tf_save_model, load_model as tf_load_model
from tensorflow.keras.saving import register_keras_serializable

# Register the custom layer to make it serializable
@register_keras_serializable(package='custom', name='EnhancedFusionLayer')
class EnhancedFusionLayer(Layer):
    def __init__(self, num_heads, key_dim, **kwargs):
        super(EnhancedFusionLayer, self).__init__(**kwargs)
        self.num_heads = num_heads  # Store num_heads as an attribute
        self.key_dim = key_dim      # Store key_dim as an attribute
        self.attention = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)
        
    def call(self, inputs):
        # Concatenate inputs along the last axis
        concatenated_inputs = Concatenate()(inputs)
        # Apply multi-head attention to concatenated inputs
        attention_output = self.attention(concatenated_inputs, concatenated_inputs)
        # Add the original concatenated inputs to the attention output
        return Add()([concatenated_inputs, attention_output])
        
    def get_Config(self):
        # Retrieve base Config and update with num_heads and key_dim
        Config = super(EnhancedFusionLayer, self).get_Config()
        Config.update({
            "num_heads": self.num_heads,  # Use stored attribute
            "key_dim": self.key_dim       # Use stored attribute
         })
        return Config

class Amber_RF:
    def __init__(self, row_hidden, col_hidden, num_classes):
        self.row_hidden = row_hidden
        self.col_hidden = col_hidden
        self.num_classes = num_classes
        self.model = None

    def conv_block(self, in_layer, filters, kernel_size):
        conv = Conv1D(filters=filters, kernel_size=kernel_size, padding='same')(in_layer)
        conv = BatchNormalization()(conv)
        conv = Activation('relu')(conv)
        return conv

    def lstm_pipe(self, in_layer):
        b1 = self.conv_block(in_layer, filters=64, kernel_size=3)
        b1 = MaxPooling1D(pool_size=2)(b1)
        b2 = self.conv_block(b1, filters=128, kernel_size=3)
        b2 = MaxPooling1D(pool_size=2)(b2)
        b3 = self.conv_block(b2, filters=256, kernel_size=3)
        b3 = MaxPooling1D(pool_size=2)(b3)
        encoded_rows = Bidirectional(LSTM(self.row_hidden, return_sequences=True))(b3)
        return LSTM(self.col_hidden)(encoded_rows)

    def build_model(self, num_features, input_shape, num_heads=4, key_dim=64):
        input_layers = []
        lstm_outputs = []

        for i in range(num_features):
            input_layer = Input(shape=input_shape, name=f'input_feature_{i+1}')
            input_layers.append(input_layer)
            lstm_output = self.lstm_pipe(Permute(dims=(1, 2))(input_layer))
            lstm_output_reshaped = Reshape((-1,))(lstm_output)
            lstm_outputs.append(lstm_output_reshaped)

        attention_outputs = []
        for i, lstm_output in enumerate(lstm_outputs):
            lstm_output_reshaped = Reshape((-1, lstm_output.shape[-1]))(lstm_output)
            attention_output = Attention()([lstm_output_reshaped, lstm_output_reshaped])
            attention_outputs.append(attention_output)

        fused_features = EnhancedFusionLayer(num_heads=num_heads, key_dim=key_dim)(attention_outputs)

        dense_output = Dense(128, activation='relu', kernel_regularizer=l2(0.0001))(fused_features)
        dense_output = BatchNormalization()(dense_output)
        dense_output = Dropout(0.1)(dense_output)
        prediction = Dense(self.num_classes, activation='softmax')(GlobalMaxPooling1D()(dense_output))

        self.model = Model(inputs=input_layers, outputs=prediction)

    def compile_model(self):
        optimizer = RMSprop(learning_rate=0.00001)
        self.model.compile(optimizer=optimizer, loss='mean_squared_error', metrics=['accuracy'])

    def train_model(self, X_train_list, y_train, X_val_list, y_val, epochs=30, batch_size=32):
        reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=0.00001, verbose=1)
        early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

        history = self.model.fit(
            X_train_list, y_train,
            epochs=2,
            batch_size=64,
            validation_data=(X_val_list, y_val),
            verbose=1,
            callbacks=[reduce_lr, early_stopping]
        )
        return history

    def save_model(self, path):
        """Save the model, including custom layers, to a .keras file."""
        # Save the model architecture and weights
        self.model.save(path)
        
        # Extract custom layer metadata from the existing layers in the model
        custom_objects_metadata = {}
        for layer in self.model.layers:
            if isinstance(layer, EnhancedFusionLayer):
                custom_objects_metadata[layer.name] = layer.get_Config()

        # Save custom layers metadata (if any)
        with open(f"{path}_custom_objects.json", "w") as file:
            json.dump(custom_objects_metadata, file)
            

    @staticmethod
    def load_model(path):
        # Load custom layer metadata
        with open(f"{path}_custom_objects.json", "r") as file:
            custom_objects_metadata = json.load(file)
        
        # Load the model architecture and weights, specifying the custom layers
        model = tf_load_model(path, custom_objects={**custom_objects_metadata})
        
        # If needed, instantiate Amber_RF with proper parameters
        # Assuming the model's class properties are fixed, e.g.:
        amber_model = Amber_RF(row_hidden=64, col_hidden=64, num_classes=3)
        amber_model.model = model  # Assign the loaded model to amber_model
        
        return amber_model


    def evaluate_model(self, X_test, y_test, batch_size=64):
        return self.model.evaluate(X_test, y_test, batch_size=Config.batch_size)

    def predict(self, X):
        return self.model.predict(X)

    def architecture(self):
        return self.model.summary()

In [67]:
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import (classification_report, accuracy_score, f1_score,
                             cohen_kappa_score, matthews_corrcoef, confusion_matrix)
import matplotlib.pyplot as plt
import os
import seaborn as sns

class KFoldCrossValidation:
    def __init__(self, ts_model, X_train, y_train, batch_size=64, epochs=2, k=1, save_dir='plots'):
        self.ts_model = ts_model
        self.X_train = X_train
        self.y_train = y_train
        self.batch_size = batch_size
        self.epochs = epochs
        self.k = k
        self.save_dir = save_dir
        os.makedirs(self.save_dir, exist_ok=True)  # Create directory if it doesn't exist
        self.history_accumulated = {"accuracy": [], "loss": [], "val_accuracy": [], "val_loss": []}  # Initialize empty dictionaries to accumulate metrics
        self.fold_history = []  # Initialize list to store individual fold histories

    def plot_confusion_matrix(self, fold, confusion_mat):
        plt.figure(figsize=(8, 6))
        sns.heatmap(confusion_mat, annot=True, fmt='d', cmap='Blues', cbar=False)
        plt.title(f'Confusion Matrix - Fold {fold + 1}')
        plt.xlabel('Predicted label')
        plt.ylabel('True label')
        plt.savefig(os.path.join(self.save_dir, f'confusion_matrix_fold_{fold + 1}.png'))  # Save each plot with a unique filename
        plt.close()

    def plot_individual_metrics(self, fold):
        fig, axs = plt.subplots(1, 2, figsize=(12, 6))  # Create subplots for accuracy and loss

        # Plot training accuracy
        axs[0].plot(self.fold_history[fold]['accuracy'], label='Training Accuracy')
        axs[0].plot(self.fold_history[fold]['val_accuracy'], label='Validation Accuracy')
        axs[0].set_title('Accuracy - Fold {}'.format(fold + 1))
        axs[0].set_xlabel('Epoch')
        axs[0].set_ylabel('Accuracy')
        axs[0].legend()

        # Plot training loss
        axs[1].plot(self.fold_history[fold]['loss'], label='Training Loss')
        axs[1].plot(self.fold_history[fold]['val_loss'], label='Validation Loss')
        axs[1].set_title('Loss - Fold {}'.format(fold + 1))
        axs[1].set_xlabel('Epoch')
        axs[1].set_ylabel('Loss')
        axs[1].legend()

        plt.tight_layout()
        plt.savefig(os.path.join(self.save_dir, f'individual_metrics_fold_{fold + 1}.png'))  # Save the plot with a unique filename
        plt.close()  # Close the plot to avoid displaying it

    def plot_overall_metrics(self):
        fig, axs = plt.subplots(2, 2, figsize=(12, 10))  # Create subplots for accuracy, validation accuracy, loss, and validation loss

        # Plot overall accuracy
        for fold in range(self.k):
            axs[0, 0].plot(range(1, self.epochs + 1), self.fold_history[fold]['accuracy'], label=f'Fold {fold + 1}')
            axs[0, 0].set_title('Accuracy')
            axs[0, 0].set_xlabel('Epoch')
            axs[0, 0].set_ylabel('Accuracy')
            axs[0, 0].legend()

        # Plot overall validation accuracy
        for fold in range(self.k):
            axs[0, 1].plot(range(1, self.epochs + 1), self.fold_history[fold]['val_accuracy'], label=f'Fold {fold + 1}')
            axs[0, 1].set_title('Validation Accuracy')
            axs[0, 1].set_xlabel('Epoch')
            axs[0, 1].set_ylabel('Accuracy')
            axs[0, 1].legend()

        # Plot overall loss
        for fold in range(self.k):
            axs[1, 0].plot(range(1, self.epochs + 1), self.fold_history[fold]['loss'], label=f'Fold {fold + 1}')
            axs[1, 0].set_title('Loss')
            axs[1, 0].set_xlabel('Epoch')
            axs[1, 0].set_ylabel('Loss')
            axs[1, 0].legend()

        # Plot overall validation loss
        for fold in range(self.k):
            axs[1, 1].plot(range(1, self.epochs + 1), self.fold_history[fold]['val_loss'], label=f'Fold {fold + 1}')
            axs[1, 1].set_title('Validation Loss')
            axs[1, 1].set_xlabel('Epoch')
            axs[1, 1].set_ylabel('Loss')
            axs[1, 1].legend()

        plt.tight_layout()
        plt.savefig(os.path.join(self.save_dir, 'overall_metrics.png'))
        plt.close()  # Close the plot to avoid displaying it

    def run(self):
        kf = KFold(n_splits=self.k, shuffle=True)
        all_test_losses = []
        all_test_accuracies = []
        for fold, (train_index, test_index) in enumerate(kf.split(self.X_train[0])):
            print(f"Fold {fold + 1}/{self.k}")
            X_fold_train = [X[train_index] for X in self.X_train]
            y_fold_train = self.y_train[train_index]
            X_fold_val = [X[test_index] for X in self.X_train]
            y_fold_val = self.y_train[test_index]
            self.ts_model.build_model(num_features=3, input_shape=(Config.N_TIME_STEPS, 1))
            self.ts_model.compile_model()
            history = self.ts_model.train_model(X_fold_train, y_fold_train, X_fold_val, y_fold_val, epochs=Config.epochs, batch_size=Config.batch_size)
            test_loss, test_accuracy = self.ts_model.evaluate_model(X_fold_val, y_fold_val)
            all_test_losses.append(test_loss)
            all_test_accuracies.append(test_accuracy)
            print(f"Fold {fold + 1} - Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")

            # Accumulate history
            self.fold_history.append(history.history)

            # Generate confusion matrix
            y_pred_val = self.ts_model.predict(X_fold_val)
            y_pred_classes = np.argmax(y_pred_val, axis=1)
            y_true_classes = np.argmax(y_fold_val, axis=1)

            confusion_mat = confusion_matrix(y_true_classes, y_pred_classes)

            # Print confusion matrix
            print(f"Confusion Matrix for Fold {fold + 1}:\n{confusion_mat}\n")

            # Save confusion matrix plot
            self.plot_confusion_matrix(fold, confusion_mat)

            # Save individual plots
            self.plot_individual_metrics(fold)

        avg_test_loss = np.mean(all_test_losses)
        avg_test_accuracy = np.mean(all_test_accuracies)
        print(f"Average Test Loss: {avg_test_loss:.4f}, Average Test Accuracy: {avg_test_accuracy:.4f}")

        # Plot overall metrics
        #self.plot_overall_metrics()

        return self.history_accumulated

In [68]:
def evaluate_model_performance(model, X_test_list, y_test_reshaped):
    # Predict classes for test data
    y_pred = model.predict(X_test_list)
    y_pred_classes = np.argmax(y_pred, axis=1)
    y_true_classes = np.argmax(y_test_reshaped, axis=1)

    # Calculate classification metrics
    classification_report_str = classification_report(y_true_classes, y_pred_classes)
    accuracy = accuracy_score(y_true_classes, y_pred_classes)
    f1 = f1_score(y_true_classes, y_pred_classes, average='weighted')
    cohen_kappa = cohen_kappa_score(y_true_classes, y_pred_classes)
    mcc = matthews_corrcoef(y_true_classes, y_pred_classes)
    confusion_mat = confusion_matrix(y_true_classes, y_pred_classes)

    # Calculate various metrics
    TP = np.diag(confusion_mat)
    FP = confusion_mat.sum(axis=0) - TP
    FN = confusion_mat.sum(axis=1) - TP
    TN = confusion_mat.sum() - (TP + FP + FN)

    TPR = TP / (TP + FN)
    TNR = TN / (TN + FP)
    PPV = TP / (TP + FP)
    NPV = TN / (TN + FN)
    FPR = FP / (FP + TN)
    FNR = FN / (TP + FN)
    FDR = FP / (TP + FP)
    ACC = (TP + TN) / (TP + FP + FN + TN)

    return {
        "classification_report": classification_report_str,
        "accuracy": accuracy,
        "f1": f1,
        "cohen_kappa": cohen_kappa,
        "mcc": mcc,
        "confusion_matrix": confusion_mat,
        "TPR": TPR,
        "TNR": TNR,
        "PPV": PPV,
        "NPV": NPV,
        "FPR": FPR,
        "FNR": FNR,
        "FDR": FDR,
        "ACC": ACC
    }


In [72]:
file_path = '../Data/osdb_3min_allSeizures.json'  # Replace with your JSON file path
ipd_dataloader = IpdDataLoader(file_path, time_steps=Config.N_TIME_STEPS)
df_result = ipd_dataloader.load_and_process_data_from_json()
#xx= dataloader.df_sensordata
reshaper = IpdDataReshaper(ipd_dataloader.df_sensordata)
reshaped_df = reshaper.reshape_data()
# Initialize Interpolator and interpolate the 'hr' column
interpolator = IpdInterpolator(reshaped_df, column_to_interpolate="hr")
interpolator.interpolate_column(
            new_column_name="interpolated_hr",
            interval=Config.N_TIME_STEPS,
            time_step=Config.time_step_length,
        )
dataset_df = interpolator.get_dataframe()
# Initialize the processor with the path to the CSV file
processor = IpdDataProcessor("../Data/ipd_labels.csv")

# Filter and sort the sensor data
processor.filter_and_sort_sensor_data(dataset_df)

# Merge the label column into the filtered sensor data
processor.merge_labels()

# Retrieve the processed sensor data
processed_data = processor.get_sensor_data()

undersampler = TimeSeriesUndersampler(timestep_size=125, prefix=999)
target_counts = {0: 92500, 1: 92500, 2: 92500}
resampled_df = undersampler.fit_resample(processed_data, target_column='label', target_counts=target_counts)

# Step 6: Load data
data_loader = DataLoader(
    dataframe=resampled_df,
    time_steps=Config.N_TIME_STEPS,
    step=Config.step,
    target_column="label",
    )

df_label = data_loader.load_data()
df_label.head()


# Step 7: Split by event
splitter = EventBasedSplitter(dataframe=df_label, test_size=0.25)
train_df, test_df = splitter.split_by_event()

# Extract training and testing data
X_train, y_train = train_df.drop(columns=['label']), train_df['label']
X_test, y_test = test_df.drop(columns=['label']), test_df['label']

# Reshape data for the model
X_train_reshaped = {
            "feature_1": X_train["feature_1"].values,
            "feature_2": X_train["feature_2"].values,
            "feature_3": X_train["feature_3"].values,
}
X_test_reshaped = {
            "feature_1": X_test["feature_1"].values,
            "feature_2": X_test["feature_2"].values,
            "feature_3": X_test["feature_3"].values,
}
y_test_reshaped = y_test.values.astype(np.float32)

# Step 8: Initialize model and perform training with cross-validation
ts_model = Amber_RF(row_hidden=Config.row_hidden, col_hidden=Config.row_hidden, num_classes=3)

# Perform K-Fold Cross Validation
kfold_cv = KFoldCrossValidation(
    ts_model,
    [X_train_reshaped['feature_1'], X_train_reshaped['feature_2'], X_train_reshaped['feature_3']],
    y_train,
    )

kfold_cv.run()

# Evaluate the model
evaluation_results = evaluate_model_performance(
    ts_model,
    [X_test_reshaped['feature_1'], X_test_reshaped['feature_2'], X_test_reshaped['feature_3']],
    y_test_reshaped,
    )

print("\nOverall Classification Results\n")
print("Accuracy:", evaluation_results["accuracy"])
print("F1 Score:", evaluation_results["f1"])

Interpolation completed. New column 'interpolated_hr' added to the DataFrame.


C:\Users\jamie\AppData\Local\Temp\ipykernel_13992\2245860014.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.df_sensor_data_filtered['eventId'] = pd.Categorical(self.df_sensor_data_filtered['eventId'], categories=self.ids, ordered=True)
C:\Users\jamie\AppData\Local\Temp\ipykernel_13992\249430101.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  timestep['eventId'] = new_event_id
C:\Users\jamie\AppData\Local\Temp\ipykernel_13992\2074117667.py:26: FutureWarning: Unlike other reduction functions (

KeyError: 'feature_1'

In [13]:
ipd_dataloader

,eventId,userId,hr,rawData,FFT
0,407,39,67,"[1496, 1480, 1500, 1492, 1496, 1484, 1500, 149...","[1.2960299500264227e-11, 143.05125737182817, 5..."
1,407,39,67,"[1492, 1508, 1496, 1476, 1484, 1476, 1496, 150...","[9.094947017729282e-13, 75.02350794818989, 31...."
2,407,39,68,"[1488, 1496, 1484, 1492, 1492, 1508, 1504, 148...","[2.2737367544323206e-13, 91.25440903139302, 81..."
3,407,39,69,"[1488, 1476, 1480, 1504, 1496, 1508, 1484, 148...","[1.3642420526593924e-11, 101.37768172754971, 7..."
4,407,39,69,"[1504, 1488, 1504, 1492, 1484, 1500, 1496, 149...","[7.275957614183426e-12, 116.42740204040989, 77..."
...,...,...,...,...,...
3944,53666,39,-1,"[1001.399048, 1004.223083, 1015.488037, 994.85...","[8.640199666842818e-12, 73.47271094056933, 72...."
3945,53666,39,-1,"[982.698303, 987.676086, 1013.998047, 1001.670...","[1.3301360013429075e-11, 128.37305265198444, 4..."
3946,53666,39,-1,"[999.175659, 1009.316589, 1002.189575, 988.712...","[2.0463630789890885e-12, 95.03426164728164, 88..."
3947,53666,39,-1,"[1009.403809, 999.952026, 1012.031616, 999.607...","[1.1482370609883219e-11, 140.2430826956883, 73..."


In [48]:
def evaluate_model_performance(model, X_test_list, y_test_reshaped):
    # Predict classes for test data
    y_pred = model.predict(X_test_list)
    y_pred_classes = np.argmax(y_pred, axis=1)
    y_true_classes = np.argmax(y_test_reshaped, axis=1)

    # Calculate classification metrics
    classification_report_str = classification_report(y_true_classes, y_pred_classes)
    accuracy = accuracy_score(y_true_classes, y_pred_classes)
    f1 = f1_score(y_true_classes, y_pred_classes, average='weighted')
    cohen_kappa = cohen_kappa_score(y_true_classes, y_pred_classes)
    mcc = matthews_corrcoef(y_true_classes, y_pred_classes)
    confusion_mat = confusion_matrix(y_true_classes, y_pred_classes)

    # Calculate various metrics
    TP = np.diag(confusion_mat)
    FP = confusion_mat.sum(axis=0) - TP
    FN = confusion_mat.sum(axis=1) - TP
    TN = confusion_mat.sum() - (TP + FP + FN)

    TPR = TP / (TP + FN)
    TNR = TN / (TN + FP)
    PPV = TP / (TP + FP)
    NPV = TN / (TN + FN)
    FPR = FP / (FP + TN)
    FNR = FN / (TP + FN)
    FDR = FP / (TP + FP)
    ACC = (TP + TN) / (TP + FP + FN + TN)

    return {
        "classification_report": classification_report_str,
        "accuracy": accuracy,
        "f1": f1,
        "cohen_kappa": cohen_kappa,
        "mcc": mcc,
        "confusion_matrix": confusion_mat,
        "TPR": TPR,
        "TNR": TNR,
        "PPV": PPV,
        "NPV": NPV,
        "FPR": FPR,
        "FNR": FNR,
        "FDR": FDR,
        "ACC": ACC
    }


In [73]:
class DataPreprocessingPipeline:
    def __init__(self, json_path, csv_path, Config):
        self.json_path = json_path
        self.csv_path = csv_path
        self.Config = Config

    def run(self):
        # Step 1: Load and process data
        ipd_dataloader = IpdDataLoader(self.json_path, time_steps=self.Config.N_TIME_STEPS)
        df_result = ipd_dataloader.load_and_process_data_from_json()

        # Step 2: Reshape data
        reshaper = IpdDataReshaper(ipd_dataloader.df_sensordata)
        reshaped_df = reshaper.reshape_data()

        # Step 3: Interpolate
        interpolator = IpdInterpolator(reshaped_df, column_to_interpolate="hr")
        interpolator.interpolate_column(
            new_column_name="interpolated_hr",
            interval=self.Config.N_TIME_STEPS,
            time_step=self.Config.time_step_length,
        )
        dataset_df = interpolator.get_dataframe()

        # Step 4: Process label
        processor = IpdDataProcessor(self.csv_path)
        processor.filter_and_sort_sensor_data(dataset_df)
        processor.merge_label()
        processed_data = processor.get_sensor_data()

        # Step 5: Undersample
        undersampler = TimeSeriesUndersampler(
            dataframe=processed_data,
            rows_per_group=self.Config.N_TIME_STEPS,
            target_class_count=self.Config.target_class_count,
        )
        undersampler.undersample_data()
        undersampler.assign_ids_and_structure(user_id_start=1, event_id_start=1)
        final_df = undersampler.get_final_dataframe()

        # Step 6: Load data
        data_loader = DataLoader(
            dataframe=final_df,
            time_steps=self.Config.N_TIME_STEPS,
            step=self.Config.step,
            target_column="label",
        )
        df_label = data_loader.load_data()

        # Step 7: Split by event
        splitter = EventBasedSplitter(dataframe=df_label, test_size=0.25)
        train_df, test_df = splitter.split_by_event()

        # Extract training and testing data
        X_train, y_train = train_df.drop(columns=['label']), train_df['label']
        X_test, y_test = test_df.drop(columns=['label']), test_df['label']

        # Reshape data for the model
        X_train_reshaped = {
            "Feature_1": X_train["feature_1"].values,
            "Feature_2": X_train["feature_2"].values,
            "Feature_3": X_train["feature_3"].values,
        }
        X_test_reshaped = {
            "Feature_1": X_test["feature_1"].values,
            "Feature_2": X_test["feature_2"].values,
            "Feature_3": X_test["feature_3"].values,
        }
        y_test_reshaped = y_test.values.astype(np.float32)

        # Step 8: Initialize model and perform training with cross-validation
        ts_model = Amber_RF(row_hidden=self.Config.row_hidden, col_hidden=self.Config.row_hidden, num_classes=2)

        # Perform K-Fold Cross Validation
        kfold_cv = KFoldCrossValidation(
            ts_model,
            [X_train_reshaped['Feature_1'], X_train_reshaped['Feature_2'], X_train_reshaped['Feature_3']],
            y_train,
        )
        kfold_cv.run()

        # Evaluate the model
        evaluation_results = evaluate_model_performance(
            ts_model,
            [X_test_reshaped['Feature_1'], X_test_reshaped['Feature_2'], X_test_reshaped['Feature_3']],
            y_test_reshaped,
        )

        print("\nOverall Classification Results\n")
        print("Accuracy:", evaluation_results["accuracy"])
        print("F1 Score:", evaluation_results["f1"])

        return train_df, test_df, evaluation_results


# Example usage
if __name__ == "__main__":
    Config = Config()  # Define your Configuration object
    pipeline = DataPreprocessingPipeline(
        json_path="../Data/osdb_3min_allSeizures.json",
        csv_path="../Data/ipd_labels.csv",
        Config=Config,
    )
    train_df, test_df, results = pipeline.run()
    print("Train DataFrame:", train_df.head())
    print("Test DataFrame:", test_df.head())


Interpolation completed. New column 'interpolated_hr' added to the DataFrame.


C:\Users\jamie\AppData\Local\Temp\ipykernel_13992\2245860014.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.df_sensor_data_filtered['eventId'] = pd.Categorical(self.df_sensor_data_filtered['eventId'], categories=self.ids, ordered=True)


AttributeError: 'IpdDataProcessor' object has no attribute 'merge_label'

In [21]:
class DataPreprocessingPipeline:
    def __init__(self, json_path, csv_path, Config):
        self.json_path = json_path
        self.csv_path = csv_path
        self.Config = Config

    def run(self):
        # Step 1: Load and process data
        ipd_dataloader = IpdDataLoader(self.json_path, time_steps=self.Config.N_TIME_STEPS)
        df_result = ipd_dataloader.load_and_process_data_from_json()

        # Step 2: Reshape data
        reshaper = IpdDataReshaper(ipd_dataloader.df_sensordata)
        reshaped_df = reshaper.reshape_data()

        # Step 3: Interpolate
        interpolator = IpdInterpolator(reshaped_df, column_to_interpolate="hr")
        interpolator.interpolate_column(
            new_column_name="interpolated_hr",
            interval=self.Config.N_TIME_STEPS,
            time_step=self.Config.time_step_length,
        )
        dataset_df = interpolator.get_dataframe()

        # Step 4: Process label
        processor = IpdDataProcessor(self.csv_path)
        processor.filter_and_sort_sensor_data(dataset_df)
        processor.merge_label()
        processed_data = processor.get_sensor_data()

        # Step 5: Undersample
        undersampler = TimeSeriesUndersampler(
            dataframe=processed_data,
            rows_per_group=self.Config.N_TIME_STEPS,
            target_class_count=self.Config.target_class_count,
        )
        undersampler.undersample_data()
        undersampler.assign_ids_and_structure(user_id_start=1, event_id_start=1)
        final_df = undersampler.get_final_dataframe()

        # Step 6: Load data
        data_loader = DataLoader(
            dataframe=final_df,
            time_steps=self.Config.N_TIME_STEPS,
            step=self.Config.step,
            target_column="label",
        )
        df_label = data_loader.load_data()
        
# Example usage
if __name__ == "__main__":
    Config = Config()  # Define your Configuration object
    pipeline = DataPreprocessingPipeline(
        json_path="../Data/osdb_3min_allSeizures.json",
        csv_path="../Data/ipd_labels.csv",
        Config=Config,
    )



TypeError: 'Config' object is not callable